In [1]:
import requests

# Get series information for KXHIGHNY
url = "https://api.elections.kalshi.com/trade-api/v2/series/KXHIGHNY"
response = requests.get(url)
series_data = response.json()

print(f"Series Title: {series_data['series']['title']}")
print(f"Frequency: {series_data['series']['frequency']}")
print(f"Category: {series_data['series']['category']}")

/Users/benjaminthompson/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Series Title: Highest temperature in NYC
Frequency: daily
Category: Climate and Weather


In [2]:
# Get all open markets for the KXHIGHNY series
markets_url = f"https://api.elections.kalshi.com/trade-api/v2/markets?series_ticker=KXHIGHNY&status=open"
markets_response = requests.get(markets_url)
markets_data = markets_response.json()

print(f"\nActive markets in KXHIGHNY series:")
for market in markets_data['markets']:
    print(f"- {market['ticker']}: {market['title']}")
    print(f"  Event: {market['event_ticker']}")
    print(f"  Yes Price: ${market['yes_bid_dollars']} | Volume: {market['volume_fp']}")
    print()

# Get details for a specific event if you have its ticker
if markets_data['markets']:
    # Let's get details for the first market's event
    event_ticker = markets_data['markets'][0]['event_ticker']
    event_url = f"https://api.elections.kalshi.com/trade-api/v2/events/{event_ticker}"
    event_response = requests.get(event_url)
    event_data = event_response.json()

    print(f"Event Details:")
    print(f"Title: {event_data['event']['title']}")
    print(f"Category: {event_data['event']['category']}")
    


Active markets in KXHIGHNY series:
- KXHIGHNY-26MAR26-T72: Will the **high temp in NYC** be >72° on Mar 26, 2026?
  Event: KXHIGHNY-26MAR26
  Yes Price: $0.3200 | Volume: 1167.00

- KXHIGHNY-26MAR26-T65: Will the **high temp in NYC** be <65° on Mar 26, 2026?
  Event: KXHIGHNY-26MAR26
  Yes Price: $0.0100 | Volume: 2146.00

- KXHIGHNY-26MAR26-B71.5: Will the **high temp in NYC** be 71-72° on Mar 26, 2026?
  Event: KXHIGHNY-26MAR26
  Yes Price: $0.2400 | Volume: 896.00

- KXHIGHNY-26MAR26-B69.5: Will the **high temp in NYC** be 69-70° on Mar 26, 2026?
  Event: KXHIGHNY-26MAR26
  Yes Price: $0.2200 | Volume: 799.00

- KXHIGHNY-26MAR26-B67.5: Will the **high temp in NYC** be 67-68° on Mar 26, 2026?
  Event: KXHIGHNY-26MAR26
  Yes Price: $0.1300 | Volume: 1254.00

- KXHIGHNY-26MAR26-B65.5: Will the **high temp in NYC** be 65-66° on Mar 26, 2026?
  Event: KXHIGHNY-26MAR26
  Yes Price: $0.0500 | Volume: 435.00

- KXHIGHNY-26MAR25-T58: Will the **high temp in NYC** be >58° on Mar 25, 2026?
  

In [3]:
# Get orderbook for a specific market
# Replace with an actual market ticker from the markets list
if not markets_data['markets']:
    raise ValueError("No open markets found. Try removing status=open or choose another series.")

market_ticker = markets_data['markets'][0]['ticker']
orderbook_url = f"https://api.elections.kalshi.com/trade-api/v2/markets/{market_ticker}/orderbook"

orderbook_response = requests.get(orderbook_url)
orderbook_data = orderbook_response.json()

print(f"\nOrderbook for {market_ticker}:")
print("YES BIDS:")
for price_dollars, count_fp in orderbook_data['orderbook_fp']['yes_dollars'][:5]:  # Show top 5
    print(f"  Price: ${price_dollars}, Quantity: {count_fp}")

print("\nNO BIDS:")
for price_dollars, count_fp in orderbook_data['orderbook_fp']['no_dollars'][:5]:  # Show top 5
    print(f"  Price: ${price_dollars}, Quantity: {count_fp}")


Orderbook for KXHIGHNY-26MAR26-T72:
YES BIDS:
  Price: $0.0100, Quantity: 859.00
  Price: $0.0200, Quantity: 100.00
  Price: $0.0300, Quantity: 100.00
  Price: $0.0400, Quantity: 60.00
  Price: $0.0600, Quantity: 1.00

NO BIDS:
  Price: $0.0100, Quantity: 1111.00
  Price: $0.0300, Quantity: 526.00
  Price: $0.0500, Quantity: 115.00
  Price: $0.0600, Quantity: 100.00
  Price: $0.0700, Quantity: 29.00


## Authentication

In [4]:
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

def load_private_key_from_file(file_path):
    with open(file_path, "rb") as key_file:
        private_key = serialization.load_pem_private_key(
            key_file.read(),
            password=None,  # or provide a password if your key is encrypted
            backend=default_backend()
        )
    return private_key

In [5]:
import base64
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding, rsa
from cryptography.exceptions import InvalidSignature

def sign_pss_text(private_key: rsa.RSAPrivateKey, text: str) -> str:
    message = text.encode('utf-8')
    try:
        signature = private_key.sign(
            message,
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=padding.PSS.DIGEST_LENGTH
            ),
            hashes.SHA256()
        )
        return base64.b64encode(signature).decode('utf-8')
    except InvalidSignature as e:
        raise ValueError("RSA sign PSS failed") from e

In [7]:
import requests
import datetime

current_time = datetime.datetime.now()
timestamp = current_time.timestamp()
current_time_milliseconds = int(timestamp * 1000)
timestampt_str = str(current_time_milliseconds)

private_key = load_private_key_from_file('/Users/benjaminthompson/code/keys/priv_key_2.key')

method = "GET"
base_url = 'https://demo-api.kalshi.co'
path='/trade-api/v2/portfolio/balance'

# Strip query parameters from path before signing
path_without_query = path.split('?')[0]
msg_string = timestampt_str + method + path_without_query
sig = sign_pss_text(private_key, msg_string)

headers = {
    'KALSHI-ACCESS-KEY': 'daf6c7c1-3f38-4b70-891c-584ef6aa46bf',
    'KALSHI-ACCESS-SIGNATURE': sig,
    'KALSHI-ACCESS-TIMESTAMP': timestampt_str
}

response = requests.get(base_url + path, headers=headers)

print(response.text)

{"error":{"code":"authentication_error","message":"authentication_error","details":"NOT_FOUND"}}


In [ ]:
# Retry the authenticated request using the fresh timestamp & signature computed earlier
fresh_headers = {
    "KALSHI-ACCESS-KEY": headers["KALSHI-ACCESS-KEY"],
    "KALSHI-ACCESS-TIMESTAMP": fresh_ts,
    "KALSHI-ACCESS-SIGNATURE": fresh_sig,
}

resp = requests.get(base_url + path_without_query, headers=fresh_headers)
print("status:", resp.status_code)
print(resp.text)

In [8]:
import datetime
import base64

# Diagnostics for 401 authentication (run in new cell)

print("base_url:", base_url)
print("method:", method, "path:", path_without_query)
print("ACCESS-KEY present:", headers.get("KALSHI-ACCESS-KEY") is not None)
print("sent timestamp (ms):", timestampt_str)

# check clock skew vs UTC
now_utc_ms = int(datetime.datetime.utcnow().timestamp() * 1000)
print("current UTC (ms):", now_utc_ms, "diff (ms):", now_utc_ms - int(timestampt_str))

# recompute signature for the sent timestamp and compare
msg_sent = timestampt_str + method + path_without_query
recomputed_sig = sign_pss_text(private_key, msg_sent)
print("recomputed sig equals header sig:", recomputed_sig == sig)

# produce a fresh signature using current UTC timestamp (if clock skew is the issue)
fresh_ts = str(now_utc_ms)
fresh_msg = fresh_ts + method + path_without_query
fresh_sig = sign_pss_text(private_key, fresh_msg)
print("fresh timestamp (ms):", fresh_ts)
print("fresh signature (use these headers if clock was off):")
print("  KALSHI-ACCESS-TIMESTAMP:", fresh_ts)
print("  KALSHI-ACCESS-SIGNATURE:", fresh_sig)

# Quick checklist (printed for convenience)
print("\nChecklist:")
print("- Ensure ACCESS-KEY and private key pair belong to the environment (demo vs production).")
print("- Ensure base_url matches the credentials (demo-api.kalshi.co vs api.elections.kalshi.com).")
print("- Ensure system clock is synced to UTC (small skew can cause 401).")
print("- Ensure the signed string format is exactly: <timestamp_ms><HTTP_METHOD><path_without_query> (no extra whitespace).")

base_url: https://demo-api.kalshi.co
method: GET path: /trade-api/v2/portfolio/balance
ACCESS-KEY present: True
sent timestamp (ms): 1774465843377
current UTC (ms): 1774491247664 diff (ms): 25404287
recomputed sig equals header sig: False
fresh timestamp (ms): 1774491247664
fresh signature (use these headers if clock was off):
  KALSHI-ACCESS-TIMESTAMP: 1774491247664
  KALSHI-ACCESS-SIGNATURE: vnurwMYtLN4diJGnLu9vzeIgJ3t8yCU0RK+YGowu6Wz9oRsEoTw7i2i55GMKk1HAEwQHTABZuirRvRW7hddoGh2aVUnlpddINCf/MzCvoQCNUqZHh4sU5yujVja20TDnOpGipSRPdb6ShresSqluTxI1jWq2OE6y/VkARH2oB75cJxbIjMMEJtOjopJMbZRjt5M9PgbwUVZ6/SauBhMNkUxBugJ6ZiizvBYosG258sb/7zD2d3cyiR49wf6uJ9Bf2q4Ykr72htuRPQoRxtlCtkHpvbFJqYvc7NRqCIIQd+yLbZaP9X3PnLdLJH08E+OQGIrUlJQ1JmHLh6KLvOGuJg==

Checklist:
- Ensure ACCESS-KEY and private key pair belong to the environment (demo vs production).
- Ensure base_url matches the credentials (demo-api.kalshi.co vs api.elections.kalshi.com).
- Ensure system clock is synced to UTC (small skew can cause 401)